# Cat Behaviour Video Classifier

Create, train and export a cat behaviour classifier from labelled track embeddings.

## Configure notebook

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import json
from dataclasses import InitVar, dataclass, field
from enum import IntEnum

import joblib
import numpy as np

from utils import get_video_hashes

In [ ]:
# Define constants

In [ ]:
# Define paths
DATA_ROOT = PROJECT_ROOT / "datasets" / "behaviour_dataset"
ANN_PATH = DATA_ROOT / "behaviour_labels.json"
TRACK_FILES = sorted(DATA_ROOT.glob("track-*.joblib"))

## Load and validate tracks

In [ ]:
class AnnotationStatus(IntEnum):
    COMPLETE = 0
    VIDEO_UNLABELLED = 1
    VIDEO_UNLABELLED_CAT = 2
    PARTIAL = 3
    OVERLAPPING = 4


@dataclass
class TrackInfo:
    video_path: str
    cat_id: str
    frame_hashes: list[str]
    images: dict[str, np.ndarray]
    embeddings: dict[str, np.ndarray]
    embedding_model_sha256: str
    annotations: InitVar[list[dict]]
    video_name: str = field(init=False)
    annotation_status: AnnotationStatus = field(init=False)
    behaviour_map: dict[str, str] | None = field(init=False, default=None)

    def __post_init__(self, annotations: list[dict]) -> None:
        self.video_name = Path(self.video_path).name
        self.annotation_status = self._join_annotations(annotations)

    def __len__(self) -> int:
        return len(self.frame_hashes)

    def _join_annotations(self, annotations: list[dict]) -> AnnotationStatus:
        # check if the video is unlabelled
        video_annotations = [
            a for a in annotations if a["video_name"] == self.video_name
        ]
        if not video_annotations:
            return AnnotationStatus.VIDEO_UNLABELLED

        # check for cat specific annotations
        cat_video_annotations = [
            ann for ann in video_annotations if ann["cat_id"] == self.cat_id
        ]
        if not cat_video_annotations:
            return AnnotationStatus.VIDEO_UNLABELLED_CAT

        # create hash map
        hash_map = {h: i for i, h in enumerate(get_video_hashes(self.video_path))}

        # convert annotations to index based windows
        windows = []
        for ann in cat_video_annotations:
            start = hash_map[ann["frame_hash_start"]]
            end = hash_map[ann["frame_hash_end"]]
            windows.append((start, end, ann["behaviour"]))

        # create behaviour map for each track frame
        behaviour_map = {}
        for frame_hash in self.frame_hashes:
            matches = [b for s, e, b in windows if s <= hash_map[frame_hash] <= e]
            if len(matches) > 1:
                return AnnotationStatus.OVERLAPPING
            if not matches:
                return AnnotationStatus.PARTIAL
            behaviour_map[frame_hash] = matches[0]

        # set the behaviour map
        self.behaviour_map = behaviour_map
        return AnnotationStatus.COMPLETE


# load annotations and track information
annotations = json.loads(ANN_PATH.read_text()) if ANN_PATH.exists() else []
all_tracks = [TrackInfo(**joblib.load(p), annotations=annotations) for p in TRACK_FILES]

In [ ]:
# categorise tracks based status
complete_tracks = [
    t for t in all_tracks if t.annotation_status == AnnotationStatus.COMPLETE
]
incorrectly_labelled_tracks = [
    t
    for t in all_tracks
    if t.annotation_status
    in [
        AnnotationStatus.VIDEO_UNLABELLED_CAT,
        AnnotationStatus.PARTIAL,
        AnnotationStatus.OVERLAPPING,
    ]
]
unlabelled_videos = {
    t.video_name
    for t in all_tracks
    if t.annotation_status == AnnotationStatus.VIDEO_UNLABELLED
}
print(f"Complete tracks: {len(complete_tracks)}")
print(f"Incorrectly labelled tracks: {len(incorrectly_labelled_tracks)}")
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")